# Membangun Dataset Master Kualitas Udara Jakarta untuk Forecasting PM2.5

Notebook ini digunakan untuk membangun **dataset master kualitas udara Jakarta** sebagai fondasi awal dalam proyek **forecasting PM2.5**. Fokus utama notebook ini bukan pada tahap modeling, melainkan pada proses pengumpulan, pembersihan, penyusunan, dan penggabungan data agar siap digunakan pada tahap analisis berikutnya.

Data utama yang digunakan berasal dari **SPKU/ISPU hourly lima stasiun pemantauan kualitas udara DKI Jakarta**. Data tersebut kemudian dilengkapi dengan **data cuaca hourly dari Open-Meteo Archive API** sebagai fitur eksternal. Variabel cuaca seperti suhu, kelembapan, curah hujan, tekanan udara, kecepatan angin, dan arah angin penting karena kondisi meteorologis dapat memengaruhi naik-turunnya konsentrasi polutan udara, khususnya PM2.5.

Hasil akhir dari notebook ini adalah dataset tabular yang lebih rapi, konsisten, dan siap digunakan untuk tahapan lanjutan seperti **EDA, pengecekan missing value, feature engineering, pembuatan target horizon, dan baseline modeling**.

## Peta Sumber Data dan Cakupan Pengambilan

Notebook ini menggunakan dua kelompok data utama, yaitu **data kualitas udara SPKU/ISPU** dan **data cuaca hourly**.

Data SPKU diperoleh dari halaman detail kualitas udara untuk lima stasiun utama di DKI Jakarta. Setiap stasiun memiliki identitas berupa `station_id`, `station_slug`, dan `station_name` yang digunakan untuk membangun URL scraping serta menjaga konsistensi saat data digabungkan.

| Kode Stasiun | Nama Stasiun | Slug |
|---|---|---|
| DKI1 | Bundaran HI | `dki1-bundaran-hi` |
| DKI2 | Kelapa Gading | `dki2-kelapa-gading` |
| DKI3 | Jagakarsa | `dki3-jagakarsa` |
| DKI4 | Lubang Buaya | `dki4-lubang-buaya` |
| DKI5 | Kebun Jeruk | `dki5-kebun-jeruk` |

Rentang waktu aktual yang digunakan pada konfigurasi notebook adalah:

| Komponen | Nilai |
|---|---|
| Tanggal mulai | `2022-10-01` |
| Tanggal akhir | `2026-04-21` |
| Resolusi data | Hourly / per jam |
| Jumlah stasiun | 5 stasiun |

Catatan penting: pada komentar awal kode scraper masih terdapat keterangan rentang lama `2010-01-01 s.d. 2025-08-31`. Namun, rentang yang benar-benar digunakan oleh kode adalah nilai pada variabel `START_DATE` dan `END_DATE`, yaitu `2022-10-01` sampai `2026-04-21`.

Selain data SPKU, notebook juga mengambil data cuaca dari Open-Meteo. Data cuaca ini disesuaikan dengan koordinat masing-masing stasiun agar fitur meteorologis yang digunakan tetap relevan terhadap lokasi pengamatan kualitas udara.

## Dari Scraping sampai Dataset Siap Analisis

Alur kerja notebook ini disusun secara bertahap agar proses pengumpulan data lebih aman, mudah dicek, dan bisa dilanjutkan kembali jika sempat terhenti. Secara umum, pipeline notebook ini dapat diringkas sebagai berikut:

<div style="display:flex; flex-direction:column; align-items:center; gap:10px; margin:20px 0;">

  <div style="background:#E8F3FF; color:#0B3954; padding:12px 18px; border-radius:12px; width:75%; text-align:center; font-weight:600; border:1px solid #B8D8F8;">
    1. Cek koneksi sumber data
  </div>

  <div style="font-size:24px; color:#888;">↓</div>

  <div style="background:#EAF8EF; color:#1B5E20; padding:12px 18px; border-radius:12px; width:75%; text-align:center; font-weight:600; border:1px solid #BFE6C8;">
    2. Scraping SPKU per stasiun dan per tahun
  </div>

  <div style="font-size:24px; color:#888;">↓</div>

  <div style="background:#FFF7E6; color:#7A4B00; padding:12px 18px; border-radius:12px; width:75%; text-align:center; font-weight:600; border:1px solid #F3D79B;">
    3. Simpan data mentah SPKU ke folder raw
  </div>

  <div style="font-size:24px; color:#888;">↓</div>

  <div style="background:#F3EFFF; color:#3E2A78; padding:12px 18px; border-radius:12px; width:75%; text-align:center; font-weight:600; border:1px solid #D4C6FF;">
    4. Gabungkan file SPKU menjadi master SPKU
  </div>

  <div style="font-size:24px; color:#888;">↓</div>

  <div style="background:#EFFFFA; color:#00695C; padding:12px 18px; border-radius:12px; width:75%; text-align:center; font-weight:600; border:1px solid #B2DFDB;">
    5. Download data cuaca hourly per stasiun
  </div>

  <div style="font-size:24px; color:#888;">↓</div>

  <div style="background:#FFF0F5; color:#7B1B45; padding:12px 18px; border-radius:12px; width:75%; text-align:center; font-weight:600; border:1px solid #F5BFD3;">
    6. Gabungkan SPKU master dengan weather master
  </div>

  <div style="font-size:24px; color:#888;">↓</div>

  <div style="background:#F0F4F8; color:#263238; padding:12px 18px; border-radius:12px; width:75%; text-align:center; font-weight:700; border:1px solid #CBD5E1;">
    7. Simpan dataset master akhir
  </div>

</div>

Strategi scraping dilakukan **per stasiun dan per tahun**, bukan langsung seluruh data sekaligus. Pendekatan ini membuat proses lebih aman karena jika scraping gagal di tengah jalan, notebook tidak harus mengulang semua data dari awal.

File yang sudah berhasil dibuat dapat dilewati menggunakan pengaturan berikut:

`SKIP_EXISTING_YEAR_FILE = True`

Dengan cara ini, notebook menjadi lebih efisien ketika dijalankan ulang. Output scraping juga lebih mudah diperiksa karena setiap file masih memiliki identitas stasiun dan tahun yang jelas.

## Peran Dataset Master dalam Pipeline Forecasting

Output utama notebook ini adalah **dataset master gabungan SPKU dan weather**. Setiap baris merepresentasikan satu observasi pada satu waktu tertentu untuk satu stasiun tertentu.

Dataset akhir berisi beberapa kelompok informasi utama:

| Kelompok Informasi | Contoh Kolom | Kegunaan |
|---|---|---|
| Waktu | `datetime`, `date`, `year`, `month`, `day`, `hour_num` | Menjelaskan kapan observasi terjadi |
| Identitas stasiun | `station_id`, `station_slug`, `station_name`, `lokasi` | Menjelaskan lokasi pengamatan |
| Polutan | `pm25`, `pm10`, `so2`, `co`, `o3`, `no2`, `hc` | Variabel kualitas udara |
| Kategori ISPU | `kategori` | Label kondisi kualitas udara |
| Metadata sumber | `last_update`, `source_url`, `source_file` | Melacak asal data |
| Cuaca | `temperature_2m`, `relative_humidity_2m`, `precipitation`, `rain`, `surface_pressure`, `wind_speed_10m`, `wind_direction_10m` | Fitur eksternal meteorologis |

Dataset ini belum menjadi dataset final untuk modeling karena belum dilakukan pembuatan target horizon, lag feature, rolling feature, imputasi, dan pembagian train-test berbasis waktu. Namun, notebook ini sudah menyelesaikan tahap penting sebagai **data foundation** untuk forecasting PM2.5.

Tahap berikutnya yang dapat dilakukan setelah notebook ini adalah:

1. Mengecek kelengkapan data per stasiun.
2. Menganalisis missing value pada `pm25`.
3. Membuat target prediksi untuk horizon tertentu, misalnya H+6, H+12, atau H+24 jam.
4. Membuat fitur lag dan rolling statistik.
5. Menyusun baseline model forecasting.

## Gerbang Awal: Mengecek Akses ke Sumber Data

Cell ini digunakan untuk memastikan bahwa sumber data atau host utama dapat diakses dari environment notebook sebelum proses scraping dan download data dijalankan.

Kode melakukan request sederhana ke beberapa URL berikut.

| URL | Tujuan Pengecekan |
|---|---|
| `https://satudata.jakarta.go.id` | Mengecek akses ke portal data Jakarta |
| `https://archive-api.open-meteo.com/v1/archive` | Mengecek akses ke Open-Meteo Archive API |
| `https://data.bmkg.go.id/prakiraan-cuaca/` | Mengecek akses ke sumber data BMKG |
| `https://katalog.data.go.id` | Mengecek akses ke portal katalog data nasional |

Jika request berhasil, notebook akan menampilkan status `[OK]`.

Jika request gagal, notebook akan menampilkan status `[FAIL]` beserta jenis error yang terjadi.

Cell ini tidak menghasilkan dataset, tetapi berfungsi sebagai **diagnostic check**. Jika salah satu host gagal diakses, penyebabnya bisa berasal dari koneksi internet, DNS, pembatasan akses server, atau environment notebook yang sedang digunakan.

In [ ]:
import requests

hosts = [
    "https://satudata.jakarta.go.id",
    "https://archive-api.open-meteo.com/v1/archive",
    "https://data.bmkg.go.id/prakiraan-cuaca/",
    "https://katalog.data.go.id",
]

for url in hosts:
    try:
        r = requests.get(url, timeout=20)
        print(f"[OK] {url} -> {r.status_code}")
    except Exception as e:
        print(f"[FAIL] {url} -> {type(e).__name__}: {e}")

## Mesin Scraping SPKU: Mengambil Data Kualitas Udara per Jam

Cell ini merupakan bagian inti untuk mengambil data kualitas udara dari halaman detail SPKU/ISPU Jakarta. Data diambil berdasarkan kombinasi **stasiun** dan **tanggal**, kemudian disimpan secara bertahap dalam format file tahunan.

### Konfigurasi Utama

Beberapa konfigurasi penting dalam cell ini adalah sebagai berikut.

| Parameter | Fungsi |
|---|---|
| `START_DATE` | Menentukan tanggal awal scraping |
| `END_DATE` | Menentukan tanggal akhir scraping |
| `MAX_WORKERS` | Mengatur jumlah proses paralel saat mengambil data |
| `MAX_RETRIES` | Mengatur jumlah percobaan ulang jika request gagal |
| `TIMEOUT` | Mengatur batas waktu tunggu request |
| `DROP_ALL_NAN_ROWS` | Menentukan apakah baris yang semua polutannya kosong akan dibuang |
| `SKIP_EXISTING_YEAR_FILE` | Menentukan apakah file tahunan yang sudah ada akan dilewati |
| `RAW_DIR` | Folder penyimpanan data mentah hasil scraping |
| `LOG_DIR` | Folder penyimpanan log scraping |

Rentang waktu aktual yang digunakan adalah `2022-10-01` sampai `2026-04-21`.

### Data yang Diambil

Data polutan yang diproses dalam scraper adalah `pm10`, `pm25`, `so2`, `co`, `o3`, `no2`, dan `hc`.

Nilai kosong seperti `-`, string kosong, `nan`, dan `None` diubah menjadi missing value agar lebih aman saat dianalisis menggunakan pandas.

### Fungsi-Fungsi Penting

| Fungsi | Peran |
|---|---|
| `daterange()` | Membuat daftar tanggal dari tanggal awal sampai akhir |
| `build_url()` | Membentuk URL scraping berdasarkan stasiun dan tanggal |
| `norm_col()` | Menstandarkan nama kolom hasil parsing tabel |
| `parse_location_and_update()` | Mengambil informasi lokasi dan waktu update dari halaman |
| `parse_hourly_table()` | Mengubah tabel HTML menjadi DataFrame yang rapi |
| `fetch_one_day()` | Mengambil data satu hari untuk satu stasiun |
| `scrape_station_year()` | Mengambil data satu stasiun untuk satu tahun |
| `combine_all_yearly_files()` | Menyediakan fungsi tambahan untuk menggabungkan file tahunan ke format parquet |

### Metadata yang Ditambahkan

Setiap data hasil scraping dilengkapi metadata berikut.

| Kolom Metadata | Fungsi |
|---|---|
| `date` | Tanggal observasi |
| `datetime` | Waktu observasi lengkap sampai level jam |
| `station_id` | ID stasiun |
| `station_slug` | Slug stasiun |
| `station_name` | Nama stasiun |
| `lokasi` | Lokasi stasiun dari halaman sumber |
| `last_update` | Waktu update terakhir dari halaman sumber |
| `source_url` | URL asal data |

Metadata ini penting agar setiap baris data dapat ditelusuri kembali ke stasiun, tanggal, dan halaman sumbernya.

### Mekanisme Retry

Pada proses scraping, request tidak langsung dianggap gagal jika terjadi error. Kode akan mencoba ulang request hingga batas `MAX_RETRIES`.

Mekanisme retry ini penting karena scraping data dari web dapat mengalami gangguan sementara, seperti koneksi lambat, timeout, atau respons server yang tidak stabil.

### Mekanisme Skip File

Pada output notebook terlihat banyak pesan seperti `[SKIP] dki1-bundaran-hi_2022.csv.gz sudah ada`.

Artinya file untuk stasiun dan tahun tersebut sudah tersedia di folder output, sehingga notebook tidak melakukan scraping ulang. Ini terjadi karena konfigurasi `SKIP_EXISTING_YEAR_FILE` bernilai `True`.

Mekanisme ini membuat notebook lebih efisien ketika dijalankan ulang.

In [1]:
# =========================================================
# SCRAPER SPKU / ISPU HOURLY 5 STASIUN JAKARTA
# Range: 2010-01-01 s.d. 2025-08-31
# Output:
#   spku_hourly_raw/
#       dki1-bundaran-hi_2010.csv.gz
#       dki1-bundaran-hi_2011.csv.gz
#       ...
#       dki5-kebun-jeruk_2025.csv.gz
#   spku_hourly_logs/
#       dki1-bundaran-hi_2010_log.csv
#       ...
#   spku_hourly_5stations_2010-01-01_2025-08-31.parquet
# =========================================================

import re
import time
import random
from io import StringIO
from pathlib import Path
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests
import pandas as pd
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

# -----------------------------
# CONFIG
# -----------------------------
START_DATE = "2022-10-01"
END_DATE   = "2026-04-21"

# max_workers jangan terlalu besar biar tidak terlalu agresif ke server
MAX_WORKERS = 3

# retry/backoff
MAX_RETRIES = 3
TIMEOUT = 40
BACKOFF_BASE = 2.0

# jika True, baris jam yang semua polutannya kosong akan dibuang
DROP_ALL_NAN_ROWS = False

# kalau file tahunan sudah ada, skip
SKIP_EXISTING_YEAR_FILE = True

RAW_DIR = Path("spku_hourly_raw")
LOG_DIR = Path("spku_hourly_logs")
RAW_DIR.mkdir(exist_ok=True)
LOG_DIR.mkdir(exist_ok=True)

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "id-ID,id;q=0.9,en-US;q=0.8,en;q=0.7",
}

# 5 stasiun DKI
STATIONS = [
    {"station_id": 4, "slug": "dki1-bundaran-hi",   "station_name": "DKI1 Bundaran HI"},
    {"station_id": 5, "slug": "dki2-kelapa-gading", "station_name": "DKI2 Kelapa Gading"},
    {"station_id": 6, "slug": "dki3-jagakarsa",     "station_name": "DKI3 Jagakarsa"},
    {"station_id": 7, "slug": "dki4-lubang-buaya",  "station_name": "DKI4 Lubang Buaya"},
    {"station_id": 8, "slug": "dki5-kebun-jeruk",   "station_name": "DKI5 Kebun Jeruk"},
]

POLLUTANT_COLS = ["pm10", "pm25", "so2", "co", "o3", "no2", "hc"]


# -----------------------------
# HELPERS
# -----------------------------
def daterange(start_dt, end_dt):
    cur = start_dt
    while cur <= end_dt:
        yield cur
        cur += timedelta(days=1)

def build_url(station_id, slug, dt):
    return f"https://rendahemisi.jakarta.go.id/ispu-detail/{station_id}/{slug}/{dt.strftime('%d-%m-%Y')}"

def norm_col(c):
    c = str(c).strip().lower()
    c = c.replace(" ", "").replace(".", "")
    return c

def extract_regex(text, pattern, flags=re.S):
    m = re.search(pattern, text, flags)
    return m.group(1).strip() if m else None

def parse_location_and_update(page_text):
    # Ambil teks di antara label-label utama
    last_update = extract_regex(
        page_text,
        r"Terakhir Update:\s*(.*?)\s*(?:Lokasi:|Sumber:|Keterangan:|$)"
    )
    lokasi = extract_regex(
        page_text,
        r"Lokasi:\s*(.*?)\s*(?:Sumber:|Keterangan:|$)"
    )
    return lokasi, last_update

def parse_hourly_table(html, station, dt, url):
    """
    Return DataFrame atau None kalau tabel tidak ada.
    """
    soup = BeautifulSoup(html, "html.parser")
    page_text = soup.get_text("\n", strip=True)

    try:
        tables = pd.read_html(StringIO(html))
    except Exception:
        return None

    target = None
    for tbl in tables:
        cols = [norm_col(c) for c in tbl.columns]
        if "waktu" in cols:
            target = tbl.copy()
            target.columns = cols
            break

    if target is None:
        return None

    rename_map = {
        "waktu": "hour",
        "pm10": "pm10",
        "pm25": "pm25",
        "so2": "so2",
        "co": "co",
        "o3": "o3",
        "no2": "no2",
        "hc": "hc",
        "kategori": "kategori",
    }
    target = target.rename(columns=rename_map)

    keep_cols = ["hour", "pm10", "pm25", "so2", "co", "o3", "no2", "hc", "kategori"]
    target = target[[c for c in keep_cols if c in target.columns]].copy()

    # cleaning numeric
    for col in POLLUTANT_COLS:
        if col in target.columns:
            target[col] = (
                target[col]
                .astype(str)
                .str.strip()
                .replace({"-": pd.NA, "": pd.NA, "nan": pd.NA, "None": pd.NA})
            )
            target[col] = pd.to_numeric(target[col], errors="coerce")

    if "kategori" in target.columns:
        target["kategori"] = (
            target["kategori"]
            .astype(str)
            .str.strip()
            .replace({"-": pd.NA, "": pd.NA, "nan": pd.NA, "None": pd.NA})
        )

    # metadata
    lokasi, last_update = parse_location_and_update(page_text)

    target["date"] = pd.to_datetime(dt.date())
    target["station_id"] = station["station_id"]
    target["station_slug"] = station["slug"]
    target["station_name"] = station["station_name"]
    target["lokasi"] = lokasi
    target["last_update"] = last_update
    target["source_url"] = url

    # datetime
    target["datetime"] = pd.to_datetime(
        dt.strftime("%d-%m-%Y") + " " + target["hour"].astype(str),
        format="%d-%m-%Y %H:%M",
        errors="coerce"
    )

    # drop row jam yang semua polutannya kosong
    if DROP_ALL_NAN_ROWS:
        existing_pollutants = [c for c in POLLUTANT_COLS if c in target.columns]
        if existing_pollutants:
            target = target[target[existing_pollutants].notna().any(axis=1)].copy()

    # urutan kolom
    front = [
        "datetime", "date", "hour",
        "station_id", "station_slug", "station_name", "lokasi",
        "pm25", "pm10", "so2", "co", "o3", "no2", "hc",
        "kategori", "last_update", "source_url"
    ]
    front = [c for c in front if c in target.columns]
    other = [c for c in target.columns if c not in front]
    target = target[front + other]

    # drop duplicate kalau ada
    target = target.drop_duplicates(subset=["datetime", "station_slug"], keep="first").reset_index(drop=True)
    return target

def fetch_one_day(station, dt):
    """
    Return dict:
    {
      'date': 'YYYY-MM-DD',
      'station_slug': ...,
      'status': ...,
      'rows': int,
      'url': ...,
      'error': ...,
      'df': DataFrame or None
    }
    """
    url = build_url(station["station_id"], station["slug"], dt)

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
            status_code = resp.status_code

            if status_code == 404:
                return {
                    "date": dt.strftime("%Y-%m-%d"),
                    "station_slug": station["slug"],
                    "status": "http_404",
                    "rows": 0,
                    "url": url,
                    "error": None,
                    "df": None,
                }

            if status_code != 200:
                if attempt < MAX_RETRIES:
                    sleep_s = BACKOFF_BASE ** attempt + random.uniform(0.2, 0.8)
                    time.sleep(sleep_s)
                    continue
                return {
                    "date": dt.strftime("%Y-%m-%d"),
                    "station_slug": station["slug"],
                    "status": f"http_{status_code}",
                    "rows": 0,
                    "url": url,
                    "error": None,
                    "df": None,
                }

            df = parse_hourly_table(resp.text, station, dt, url)

            if df is None:
                return {
                    "date": dt.strftime("%Y-%m-%d"),
                    "station_slug": station["slug"],
                    "status": "no_table",
                    "rows": 0,
                    "url": url,
                    "error": None,
                    "df": None,
                }

            return {
                "date": dt.strftime("%Y-%m-%d"),
                "station_slug": station["slug"],
                "status": "ok" if len(df) > 0 else "empty_table",
                "rows": int(len(df)),
                "url": url,
                "error": None,
                "df": df,
            }

        except Exception as e:
            if attempt < MAX_RETRIES:
                sleep_s = BACKOFF_BASE ** attempt + random.uniform(0.2, 0.8)
                time.sleep(sleep_s)
                continue

            return {
                "date": dt.strftime("%Y-%m-%d"),
                "station_slug": station["slug"],
                "status": "error",
                "rows": 0,
                "url": url,
                "error": repr(e),
                "df": None,
            }

def scrape_station_year(station, year, start_dt, end_dt, max_workers=MAX_WORKERS):
    year_start = max(start_dt, datetime(year, 1, 1))
    year_end = min(end_dt, datetime(year, 12, 31))
    if year_start > year_end:
        return None

    out_csv = RAW_DIR / f"{station['slug']}_{year}.csv.gz"
    out_log = LOG_DIR / f"{station['slug']}_{year}_log.csv"

    if out_csv.exists() and SKIP_EXISTING_YEAR_FILE:
        print(f"[SKIP] {out_csv.name} sudah ada")
        return out_csv

    dates = list(daterange(year_start, year_end))
    results = []
    dfs = []

    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = {ex.submit(fetch_one_day, station, dt): dt for dt in dates}

        for fut in tqdm(as_completed(futures), total=len(futures), desc=f"{station['slug']} {year}"):
            res = fut.result()
            results.append({
                "date": res["date"],
                "station_slug": res["station_slug"],
                "status": res["status"],
                "rows": res["rows"],
                "url": res["url"],
                "error": res["error"],
            })
            if res["df"] is not None and len(res["df"]) > 0:
                dfs.append(res["df"])

    # simpan log per tahun
    log_df = pd.DataFrame(results).sort_values(["date", "station_slug"]).reset_index(drop=True)
    log_df.to_csv(out_log, index=False)

    # simpan data per tahun
    if dfs:
        year_df = pd.concat(dfs, ignore_index=True)
        year_df = year_df.sort_values(["datetime", "station_slug"]).reset_index(drop=True)
        year_df.to_csv(out_csv, index=False, compression="gzip")
        print(f"[SAVED] {out_csv} | rows={len(year_df):,}")
    else:
        # tetap simpan file kosong biar tahu sudah dicoba
        empty_cols = [
            "datetime", "date", "hour", "station_id", "station_slug", "station_name",
            "lokasi", "pm25", "pm10", "so2", "co", "o3", "no2", "hc",
            "kategori", "last_update", "source_url"
        ]
        pd.DataFrame(columns=empty_cols).to_csv(out_csv, index=False, compression="gzip")
        print(f"[SAVED EMPTY] {out_csv}")

    return out_csv

def combine_all_yearly_files(start_dt, end_dt):
    files = sorted(RAW_DIR.glob("*.csv.gz"))
    if not files:
        print("Tidak ada file tahunan untuk digabung.")
        return None

    all_df = []
    for fp in tqdm(files, desc="Combining yearly files"):
        df = pd.read_csv(fp, compression="gzip")
        if len(df) > 0:
            all_df.append(df)

    if not all_df:
        print("Semua file kosong.")
        return None

    master = pd.concat(all_df, ignore_index=True)

    # typing ulang
    if "datetime" in master.columns:
        master["datetime"] = pd.to_datetime(master["datetime"], errors="coerce")
    if "date" in master.columns:
        master["date"] = pd.to_datetime(master["date"], errors="coerce")

    for col in POLLUTANT_COLS:
        if col in master.columns:
            master[col] = pd.to_numeric(master[col], errors="coerce")

    master = master.drop_duplicates(subset=["datetime", "station_slug"], keep="first")
    master = master.sort_values(["datetime", "station_slug"]).reset_index(drop=True)

    out_parquet = Path(
        f"spku_hourly_5stations_{start_dt.strftime('%Y-%m-%d')}_{end_dt.strftime('%Y-%m-%d')}.parquet"
    )
    master.to_parquet(out_parquet, index=False)
    print(f"[MASTER SAVED] {out_parquet} | rows={len(master):,}")

    return out_parquet


# -----------------------------
# MAIN RUN
# -----------------------------
start_dt = datetime.strptime(START_DATE, "%Y-%m-%d")
end_dt   = datetime.strptime(END_DATE, "%Y-%m-%d")

for station in STATIONS:
    print(f"\n===== START {station['station_name']} =====")
    for year in range(start_dt.year, end_dt.year + 1):
        scrape_station_year(station, year, start_dt, end_dt, max_workers=MAX_WORKERS)



===== START DKI1 Bundaran HI =====
[SKIP] dki1-bundaran-hi_2022.csv.gz sudah ada
[SKIP] dki1-bundaran-hi_2023.csv.gz sudah ada
[SKIP] dki1-bundaran-hi_2024.csv.gz sudah ada
[SKIP] dki1-bundaran-hi_2025.csv.gz sudah ada
[SKIP] dki1-bundaran-hi_2026.csv.gz sudah ada

===== START DKI2 Kelapa Gading =====
[SKIP] dki2-kelapa-gading_2022.csv.gz sudah ada
[SKIP] dki2-kelapa-gading_2023.csv.gz sudah ada
[SKIP] dki2-kelapa-gading_2024.csv.gz sudah ada
[SKIP] dki2-kelapa-gading_2025.csv.gz sudah ada
[SKIP] dki2-kelapa-gading_2026.csv.gz sudah ada

===== START DKI3 Jagakarsa =====
[SKIP] dki3-jagakarsa_2022.csv.gz sudah ada
[SKIP] dki3-jagakarsa_2023.csv.gz sudah ada
[SKIP] dki3-jagakarsa_2024.csv.gz sudah ada
[SKIP] dki3-jagakarsa_2025.csv.gz sudah ada
[SKIP] dki3-jagakarsa_2026.csv.gz sudah ada

===== START DKI4 Lubang Buaya =====
[SKIP] dki4-lubang-buaya_2022.csv.gz sudah ada
[SKIP] dki4-lubang-buaya_2023.csv.gz sudah ada
[SKIP] dki4-lubang-buaya_2024.csv.gz sudah ada
[SKIP] dki4-lubang-buaya

## Menyatukan Potongan Data SPKU Menjadi Satu Master

Cell ini bertugas membaca seluruh file hasil scraping yang tersimpan di folder `spku_hourly_raw/`.

File yang dibaca memiliki format `.csv.gz`. Format tersebut adalah file CSV yang dikompresi menggunakan gzip, sehingga ukuran file lebih kecil dibandingkan CSV biasa.

### Proses yang Dilakukan

Fungsi `load_spku_master()` melakukan beberapa langkah utama:

1. Mencari semua file `.csv.gz` di folder `spku_hourly_raw/`.
2. Membaca setiap file hasil scraping.
3. Menambahkan kolom `source_file` untuk mencatat asal file.
4. Menggabungkan seluruh file menjadi satu DataFrame besar.
5. Mengubah kolom waktu menjadi format datetime.
6. Mengubah kolom polutan menjadi numerik.
7. Membuat fitur waktu tambahan.
8. Membersihkan kolom kategori.
9. Menghapus duplikasi berdasarkan `datetime` dan `station_slug`.
10. Menyimpan hasil gabungan ke file CSV.

### Fitur Waktu yang Ditambahkan

Cell ini membuat beberapa fitur turunan dari kolom `datetime`.

| Kolom Baru | Makna |
|---|---|
| `year` | Tahun observasi |
| `month` | Bulan observasi |
| `day` | Tanggal dalam bulan |
| `hour_num` | Jam observasi dalam bentuk angka |
| `dayofweek` | Hari dalam minggu, Senin = 0 |
| `is_weekend` | Penanda akhir pekan, Sabtu/Minggu = 1 |

Fitur waktu ini berguna untuk melihat pola kualitas udara berdasarkan jam, hari, bulan, atau akhir pekan.

### Output yang Dihasilkan

Berdasarkan output notebook, hasil penggabungan SPKU adalah:

| Informasi | Nilai |
|---|---|
| Shape master SPKU | `(155880, 24)` |
| Rentang waktu | `2022-10-01 00:00:00` s.d. `2026-04-21 23:00:00` |
| Jumlah stasiun | `5` |
| File output | `spku_master_hourly.csv` |

Artinya, master SPKU memiliki **155.880 baris** dan **24 kolom**. Data mencakup lima stasiun dengan rentang waktu dari **1 Oktober 2022 pukul 00:00** sampai **21 April 2026 pukul 23:00**.

File ini menjadi dataset SPKU utama sebelum digabungkan dengan data cuaca.

In [20]:
from pathlib import Path
import pandas as pd
from tqdm.auto import tqdm

RAW_DIR = Path("spku_hourly_raw")
POLLUTANT_COLS = ["pm10", "pm25", "so2", "co", "o3", "no2", "hc"]

def load_spku_master(raw_dir=RAW_DIR, save_parquet=False):
    files = sorted(raw_dir.glob("*.csv.gz"))
    if not files:
        raise FileNotFoundError(f"Tidak ada file .csv.gz di folder: {raw_dir}")

    dfs = []
    for fp in tqdm(files, desc="Loading SPKU yearly files"):
        df = pd.read_csv(fp, compression="gzip")
        if len(df) == 0:
            continue
        df["source_file"] = fp.name
        dfs.append(df)

    if not dfs:
        raise ValueError("Semua file kosong, tidak ada data yang bisa digabung.")

    master = pd.concat(dfs, ignore_index=True)

    if "datetime" in master.columns:
        master["datetime"] = pd.to_datetime(master["datetime"], errors="coerce")
    if "date" in master.columns:
        master["date"] = pd.to_datetime(master["date"], errors="coerce")

    for col in POLLUTANT_COLS:
        if col in master.columns:
            master[col] = pd.to_numeric(master[col], errors="coerce")

    master["year"] = master["datetime"].dt.year
    master["month"] = master["datetime"].dt.month
    master["day"] = master["datetime"].dt.day
    master["hour_num"] = master["datetime"].dt.hour
    master["dayofweek"] = master["datetime"].dt.dayofweek
    master["is_weekend"] = master["dayofweek"].isin([5, 6]).astype(int)

    if "kategori" in master.columns:
        master["kategori"] = (
            master["kategori"]
            .astype("string")
            .replace({"nan": pd.NA, "None": pd.NA, "": pd.NA})
        )

    master = master.drop_duplicates(subset=["datetime", "station_slug"], keep="first")
    master = master.sort_values(["station_slug", "datetime"]).reset_index(drop=True)

    print("Shape master SPKU:", master.shape)
    print("Rentang waktu:", master["datetime"].min(), "s.d.", master["datetime"].max())
    print("Jumlah stasiun:", master["station_slug"].nunique())

    master.to_csv("spku_master_hourly.csv", index=False)
    print("Saved: spku_master_hourly.csv")

    if save_parquet:
        try:
            master.to_parquet("spku_master_hourly.parquet", index=False)
            print("Saved: spku_master_hourly.parquet")
        except Exception as e:
            print("Parquet gagal, lanjut pakai CSV saja.")
            print("Error:", e)

    return master

spku_master = load_spku_master(save_parquet=False)

Loading SPKU yearly files: 100%|██████████| 25/25 [00:00<00:00, 59.61it/s]


Shape master SPKU: (155880, 24)
Rentang waktu: 2022-10-01 00:00:00 s.d. 2026-04-21 23:00:00
Jumlah stasiun: 5
Saved: spku_master_hourly.csv


## Melengkapi Data Polutan dengan Kondisi Cuaca

Cell ini digunakan untuk mengambil data cuaca hourly dari **Open-Meteo Archive API**. Data cuaca diambil untuk masing-masing lokasi stasiun SPKU menggunakan koordinat latitude dan longitude.

### Koordinat Stasiun

| Stasiun | Latitude | Longitude |
|---|---:|---:|
| DKI1 Bundaran HI | -6.1931 | 106.8230 |
| DKI2 Kelapa Gading | -6.1586 | 106.9050 |
| DKI3 Jagakarsa | -6.3346 | 106.8228 |
| DKI4 Lubang Buaya | -6.2908 | 106.9019 |
| DKI5 Kebun Jeruk | -6.1951 | 106.7694 |

### Variabel Cuaca yang Diambil

Variabel cuaca yang diambil adalah sebagai berikut.

| Variabel | Makna |
|---|---|
| `temperature_2m` | Suhu udara pada ketinggian 2 meter |
| `relative_humidity_2m` | Kelembapan relatif pada ketinggian 2 meter |
| `precipitation` | Total presipitasi |
| `rain` | Curah hujan |
| `surface_pressure` | Tekanan udara permukaan |
| `wind_speed_10m` | Kecepatan angin pada ketinggian 10 meter |
| `wind_direction_10m` | Arah angin pada ketinggian 10 meter |

Variabel cuaca ini relevan untuk forecasting PM2.5 karena kondisi meteorologis dapat memengaruhi konsentrasi polutan. Misalnya, hujan dapat menurunkan sebagian polutan melalui proses deposisi, sedangkan kecepatan dan arah angin dapat memengaruhi penyebaran polutan antarwilayah.

### Proses Download

Fungsi `fetch_openmeteo_hourly()` mengirim request ke endpoint Open-Meteo dengan beberapa parameter utama, yaitu latitude, longitude, tanggal mulai, tanggal akhir, daftar variabel hourly, dan timezone `Asia/Jakarta`.

Hasil response dari API berbentuk JSON, kemudian diubah menjadi DataFrame. Kolom `time` dari API diganti menjadi `datetime` agar formatnya konsisten dengan data SPKU.

### Output yang Dihasilkan

Data cuaca disimpan dalam dua bentuk:

1. File cuaca per stasiun di folder `weather_hourly_raw/`.
2. File gabungan seluruh stasiun bernama `weather_master_hourly.csv`.

Berdasarkan output notebook, ukuran data weather master adalah `(155880, 10)`.

Artinya, data cuaca memiliki **155.880 baris** dan **10 kolom**. Jumlah baris ini sama dengan master SPKU, sehingga data cuaca sudah siap digabungkan berdasarkan waktu dan stasiun.

In [21]:
import requests
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

STATIONS = [
    {"station_slug": "dki1-bundaran-hi",   "station_name": "DKI1 Bundaran HI",   "latitude": -6.1931, "longitude": 106.8230},
    {"station_slug": "dki2-kelapa-gading", "station_name": "DKI2 Kelapa Gading", "latitude": -6.1586, "longitude": 106.9050},
    {"station_slug": "dki3-jagakarsa",     "station_name": "DKI3 Jagakarsa",     "latitude": -6.3346, "longitude": 106.8228},
    {"station_slug": "dki4-lubang-buaya",  "station_name": "DKI4 Lubang Buaya",  "latitude": -6.2908, "longitude": 106.9019},
    {"station_slug": "dki5-kebun-jeruk",   "station_name": "DKI5 Kebun Jeruk",   "latitude": -6.1951, "longitude": 106.7694},
]

START_DATE = "2022-10-01"
END_DATE   = "2026-04-21"

HOURLY_VARS = [
    "temperature_2m",
    "relative_humidity_2m",
    "precipitation",
    "rain",
    "surface_pressure",
    "wind_speed_10m",
    "wind_direction_10m"
]

OUT_DIR = Path("weather_hourly_raw")
OUT_DIR.mkdir(exist_ok=True)

def fetch_openmeteo_hourly(lat, lon, start_date, end_date, hourly_vars):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": ",".join(hourly_vars),
        "timezone": "Asia/Jakarta"
    }
    r = requests.get(url, params=params, timeout=120)
    r.raise_for_status()
    data = r.json()

    hourly = data["hourly"]
    df = pd.DataFrame(hourly)
    df = df.rename(columns={"time": "datetime"})
    df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")
    return df

all_weather = []

for st in tqdm(STATIONS, desc="Downloading weather hourly"):
    df = fetch_openmeteo_hourly(
        lat=st["latitude"],
        lon=st["longitude"],
        start_date=START_DATE,
        end_date=END_DATE,
        hourly_vars=HOURLY_VARS
    )
    df["station_slug"] = st["station_slug"]
    df["station_name"] = st["station_name"]

    out_fp = OUT_DIR / f"{st['station_slug']}_{START_DATE}_{END_DATE}.csv"
    df.to_csv(out_fp, index=False)

    all_weather.append(df)

weather_master = pd.concat(all_weather, ignore_index=True)
weather_master = weather_master.sort_values(["station_slug", "datetime"]).reset_index(drop=True)

weather_master.to_csv("weather_master_hourly.csv", index=False)
print(weather_master.shape)
weather_master.head()

(155880, 10)


,datetime,temperature_2m,relative_humidity_2m,precipitation,rain,surface_pressure,wind_speed_10m,wind_direction_10m,station_slug,station_name
0,2022-10-01 00:00:00,26.1,82,0.0,0.0,1009.0,6.3,103,dki1-bundaran-hi,DKI1 Bundaran HI
1,2022-10-01 01:00:00,25.8,84,0.0,0.0,1008.9,6.0,123,dki1-bundaran-hi,DKI1 Bundaran HI
2,2022-10-01 02:00:00,25.2,87,0.0,0.0,1008.4,6.6,151,dki1-bundaran-hi,DKI1 Bundaran HI
3,2022-10-01 03:00:00,24.8,90,0.0,0.0,1008.2,4.7,184,dki1-bundaran-hi,DKI1 Bundaran HI
4,2022-10-01 04:00:00,24.5,92,0.0,0.0,1008.1,2.6,196,dki1-bundaran-hi,DKI1 Bundaran HI


## Menggabungkan SPKU dan Weather Menjadi Dataset Master

Cell ini menggabungkan dua dataset utama, yaitu `spku_master` dan `weather_master`.

Penggabungan dilakukan berdasarkan tiga kolom kunci:

| Kolom Kunci | Fungsi |
|---|---|
| `station_slug` | Menyamakan kode slug stasiun |
| `station_name` | Menyamakan nama stasiun |
| `datetime` | Menyamakan waktu observasi per jam |

Artinya, data cuaca akan ditempelkan ke data SPKU jika memiliki stasiun dan waktu pengamatan yang sama.

### Penyamaan Format Waktu

Sebelum proses merge, kolom `datetime` pada kedua dataset dikonversi menjadi format datetime dan dibulatkan ke level jam menggunakan `.dt.floor("H")`.

Tujuannya adalah memastikan waktu pada data SPKU dan data cuaca berada pada resolusi yang sama, yaitu per jam.

Catatan teknis: pada output muncul warning bahwa penggunaan `"H"` sudah deprecated pada pandas versi baru. Warning ini tidak menghentikan proses, tetapi untuk kode yang lebih aman ke depan, `"H"` sebaiknya diganti menjadi `"h"`.

### Strategi Merge

Notebook menggunakan strategi `how="left"`.

Dengan strategi ini, semua baris dari `spku_master` tetap dipertahankan. Data cuaca hanya ditambahkan jika pasangan `station_slug`, `station_name`, dan `datetime` ditemukan pada `weather_master`.

Strategi ini tepat karena data utama proyek adalah data kualitas udara, sedangkan data cuaca berperan sebagai fitur tambahan.

### Output Akhir

Dataset hasil merge disimpan sebagai `dataset_master_spku_weather.csv`.

Berdasarkan output cell, ukuran dataset hasil merge adalah `(155880, 31)`.

Artinya, dataset master akhir memiliki **155.880 baris** dan **31 kolom**.

Dataset ini menjadi output utama notebook dan dapat digunakan sebagai input untuk notebook lanjutan, seperti EDA, feature engineering, pembuatan target horizon, dan modeling forecasting PM2.5.

In [22]:
spku_master["datetime"] = pd.to_datetime(spku_master["datetime"]).dt.floor("H")
weather_master["datetime"] = pd.to_datetime(weather_master["datetime"]).dt.floor("H")

dataset_master = spku_master.merge(
    weather_master,
    on=["station_slug", "station_name", "datetime"],
    how="left"
)

dataset_master["date"] = dataset_master["datetime"].dt.normalize()
dataset_master.to_csv("dataset_master_spku_weather.csv", index=False)
print(dataset_master.shape)

/tmp/ipykernel_60611/3140608025.py:1: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  spku_master["datetime"] = pd.to_datetime(spku_master["datetime"]).dt.floor("H")
/tmp/ipykernel_60611/3140608025.py:2: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  weather_master["datetime"] = pd.to_datetime(weather_master["datetime"]).dt.floor("H")


(155880, 31)


## Struktur Folder dan File Output Notebook

Notebook ini menghasilkan beberapa folder dan file output. Struktur outputnya dapat diringkas sebagai berikut.

```text
project/
│
├── spku_hourly_raw/
│   ├── dki1-bundaran-hi_2022.csv.gz
│   ├── dki1-bundaran-hi_2023.csv.gz
│   ├── dki1-bundaran-hi_2024.csv.gz
│   ├── dki1-bundaran-hi_2025.csv.gz
│   ├── dki1-bundaran-hi_2026.csv.gz
│   ├── dki2-kelapa-gading_2022.csv.gz
│   ├── ...
│   └── dki5-kebun-jeruk_2026.csv.gz
│
├── spku_hourly_logs/
│   ├── dki1-bundaran-hi_2022_log.csv
│   ├── dki1-bundaran-hi_2023_log.csv
│   ├── ...
│   └── dki5-kebun-jeruk_2026_log.csv
│
├── weather_hourly_raw/
│   ├── dki1-bundaran-hi_2022-10-01_2026-04-21.csv
│   ├── dki2-kelapa-gading_2022-10-01_2026-04-21.csv
│   ├── dki3-jagakarsa_2022-10-01_2026-04-21.csv
│   ├── dki4-lubang-buaya_2022-10-01_2026-04-21.csv
│   └── dki5-kebun-jeruk_2022-10-01_2026-04-21.csv
│
├── spku_master_hourly.csv
├── weather_master_hourly.csv
└── dataset_master_spku_weather.csv
```

### Penjelasan Folder

| Folder | Isi | Fungsi |
|---|---|---|
| `spku_hourly_raw/` | File hasil scraping SPKU per stasiun dan per tahun dalam format `.csv.gz` | Menyimpan data mentah kualitas udara |
| `spku_hourly_logs/` | File log scraping per stasiun dan per tahun | Mencatat status scraping, jumlah baris, dan kemungkinan error |
| `weather_hourly_raw/` | File weather hourly per stasiun | Menyimpan data cuaca mentah dari Open-Meteo |

### Penjelasan File Output Utama

| File | Fungsi |
|---|---|
| `spku_master_hourly.csv` | Gabungan seluruh file SPKU dari folder `spku_hourly_raw/` |
| `weather_master_hourly.csv` | Gabungan seluruh data cuaca dari lima stasiun |
| `dataset_master_spku_weather.csv` | Dataset master akhir hasil merge SPKU dan weather |

File yang paling penting untuk tahap berikutnya adalah `dataset_master_spku_weather.csv`.

File ini menjadi input utama untuk proses EDA, feature engineering, pembuatan target horizon, dan baseline forecasting PM2.5.